<a href="https://colab.research.google.com/github/laurencoetzee001/african_marriage_structures/blob/main/haiku_marriage_classifier_twostage_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marriage Practice Classifier — Two-Stage Pipeline
**Google Colab | Anthropic Claude Haiku | Codebook v5**

### Codebook v5 changes (26/03/2026)
- **`3_pol_struc` simplified**: `seq_replace` (code 4) removed — sequential replacement is monogamy, not polygamy. Codes now 1–3 only. `varied` always flagged.
- **Exceptional behaviour flagging**: LLM must populate `11_conf_note` AND add `"EXCEPTIONAL"` to `12_validation_flags` for any anomalous case (polyandry, cross-ethnic marriage, religious figures, captives, author's own marriage, etc.)
- **No more `unclear` or `unspecified` codes** (inherited from v4) — LLM must commit to the best-fit code for every variable; uncertainty is captured in `12_validation_flags` instead
- Variables affected: `0_subject` (removed code 5), `2_mar_diff` (removed code 3), `3_pol_struc` (removed code 5), `5_soc_stat` (removed code 4), `6_obs_type` (removed code 5), `7_obs_location` (removed code 3), `9_moral_op` (removed code 3)
- **Human-in-the-loop (Cell 8)**: interactive Colab widget steps through all flagged rows, shows text + LLM codes, lets reviewer accept or override each flagged variable, and saves a reconciled CSV

### Inherited from v3
- `1_mar_prac` code 4 = `not_applicable` (covers contents pages, ToC, OCR failures, no marriage content)
- Monogamy = one spouse (any gender); polygamy = multiple simultaneous spouses (incl. polyandry)
- Code 3 (both) = two distinct practices for different groups in same extract
- Harem → polygamy (code 2)
- `12_validation_flags`: per-variable flag when LLM confidence < 70%

### Pipeline
- **Stage 1** (cheap): Is this snippet relevant? NO → skip. YES → Stage 2.
- **Stage 2** (full codebook): Apply all variables. Flag uncertain codes.
- **Cell 8** (human review): Step through flagged rows interactively and reconcile.

### Before running
1. Click the **🔑 key icon** → add secret `ANTHROPIC_API_KEY` → toggle Notebook access ON
2. Update `INPUT_PATH` and `OUTPUT_DIR` in Cell 2
3. Run Cells 1–7, then Cell 8 for human review

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — INSTALL & MOUNT DRIVE
# ════════════════════════════════════════════════════════

!pip install -q anthropic

print('✓ anthropic installed')
print('✓ Upload test_sample.xlsx via the Files panel (folder icon) in the left sidebar')
print('✓ Results will be saved to /content/results/')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.2 MB/s eta 0:00:00
✓ anthropic installed
✓ Upload test_sample.xlsx via the Files panel (folder icon) in the left sidebar
✓ Results will be saved to /content/results/


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ════════════════════════════════════════════════════════

from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

# ── File paths ────────────────────────────────────────────────────────────
INPUT_PATH  = '/content/test_sample.xlsx'
OUTPUT_DIR  = '/content/results'

TEXT_COLUMN = 'full_text'
MAX_ROWS    = 100        # set to None for full dataset; 100 for pilot run

# ── Model settings ────────────────────────────────────────────────────────
MODEL         = 'claude-haiku-4-5-20251001'
TEMPERATURE   = 0.2
BATCH_SIZE    = 100
REQUEST_DELAY = 1.0

assert ANTHROPIC_API_KEY, (
    'ANTHROPIC_API_KEY not found. '
    'Add it via the \U0001f511 Secrets panel in the left sidebar.'
)
print(f'✓ API key loaded')
print(f'✓ Model : {MODEL}')
print(f'✓ Input : {INPUT_PATH}')
print(f'✓ Output: {OUTPUT_DIR}')

✓ API key loaded
✓ Model : claude-haiku-4-5-20251001
✓ Input : /content/test_sample.xlsx
✓ Output: /content/results


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — EMBEDDED PROMPTS  (Codebook v4)
#
# KEY v4 RULE: no 'unclear' or 'unspecified' codes.
# LLM always commits to best-fit code + flags if <70% confident.
# ════════════════════════════════════════════════════════

STAGE_1_PROMPT = """You are a relevance filter for a historical research project on marriage practices in precolonial Africa (1500-1900), drawing on European travel accounts.

Assess whether the text extract below is applicable for marriage structure coding.

A snippet PASSES (applicable: true) only if ALL three criteria are met:
1. HUMAN SUBJECTS — the text describes human beings, not animals or objects
2. READABLE — the text is coherent and interpretable; not so corrupted by OCR errors that it cannot be understood
3. MARRIAGE CONTENT — the text contains at least one substantive reference to: marriage, wives, husbands, marital customs, polygamy, polygyny, polyandry, monogamy, harem, harems, bride-price, betrothal, or related social structures

A snippet FAILS (applicable: false) if ANY of the following apply:
- No marriage-related content whatsoever
- Text is a contents page, chapter list, or table of contents (even if it mentions relevant chapter titles)
- Text consists primarily of page numbers, headings, or navigation entries
- Text is so corrupted by OCR errors, garbling, or truncation that its meaning cannot be determined
- Text discusses only trade, geography, flora, fauna, warfare, or other topics with no marriage reference
- Text discusses animals or non-human subjects only
- A single incidental mention of \"wife\" or \"husband\" with no structural or descriptive content (e.g. \"he returned home to his wife\")

Return ONLY valid JSON. No explanation, no markdown, no text before or after:
{\"applicable\": true}
OR
{\"applicable\": false, \"reason\": \"<one short phrase>\"}"""


STAGE_2_PROMPT = """# MARRIAGE PRACTICES CODING INSTRUMENT v5.0
# Time Traveller Corpus | Precolonial Africa 1500-1900

## TASK
Classify the marriage structure described in the text extract.
- Use ONLY information present in the text. Do NOT apply external cultural or historical knowledge.
- Apply codes based on what the author reports, not what you infer about the society.
- String fields must be copied VERBATIM — preserve original spelling, transliteration, and capitalisation exactly.
- Where a conditional rule requires null, output null. Otherwise always output a code.

## CORE RULE — NO UNCERTAIN CODES
There are no 'unclear' or 'unspecified' codes in this instrument.
When you are uncertain, you MUST still select the single best-fit code based on available evidence,
AND add that variable name to 12_validation_flags to request human review.
Do NOT leave coded variables blank. Do NOT invent new codes.

---

## VARIABLE 0 — 0_subject: Who Are the Individuals in This Marriage?
Null if 1_mar_prac = 4.
Use ONLY explicit textual evidence to assign. When the text does not name the parties,
select the most plausible code given the context AND add \"0_subject\" to 12_validation_flags.

| Code | Label | Apply when |
|------|-------|------------|
| 1 | african_african | Both parties identified as African, same ethnic group or location |
| 2 | african_african_diff | Both parties African but different ethnic groups or geographic locations |
| 3 | african_foreign | One party African, one party European or foreign traveller |
| 4 | foreign_foreign | Both parties European or foreign |

When in doubt: if the extract is about an African ethnic group's customs → code 1.
If the author describes their own marriage to a local → code 3.
Always flag in 12_validation_flags if inferred rather than explicit.

---

## VARIABLE 1 — 1_mar_prac: Marriage Practice Type

| Code | Label | Apply when |
|------|-------|------------|
| 1 | monogamy_only | ONLY monogamous structure — one spouse per partner (any gender) |
| 2 | polygamy_only | ONLY polygamous structure — multiple simultaneous spouses (includes polygyny AND polyandry) |
| 3 | both | Two DISTINCT marriage practices described for DIFFERENT groups or societies within the same extract |
| 4 | not_applicable | Contents page, table of contents, OCR too corrupted to read, error, or no marriage content — ALL variables null |

STOP CONDITION: IF 1_mar_prac = 4 → set ALL variables (including 0_subject) to null and stop.

Monogamy (code 1): \"monogamy\"/\"monogamous\"/\"one wife\"/\"one husband\"/\"single spouse\" / legal prohibition of multiple spouses with no countervailing practice.
Polygamy explicit (code 2): \"polygamy\"/\"polygyny\"/\"polyandry\"/\"multiple wives\"/\"several wives\"/\"two wives\"/\"many wives\"/\"plural wives\"/\"multiple husbands\"
Polygamy implicit (code 2): \"his wives\" (plural) / ranked wives (\"first wife\"/\"senior wife\"/\"chief wife\") / \"co-wives\" / HAREM or HAREMS mentioned
Code 3 (both): Two distinct groups each described with DIFFERENT structures. E.g. \"Turks maintain harems; Greeks have but one wife.\" NOT for one group with both practices.
CRITICAL: singular \"wife\" with no structure indicated → pick best-fit code (usually 1) + flag. Harem = code 2. Divorce/remarriage without simultaneous co-wives = code 1.

---

## VARIABLE 2 — 2_mar_diff: Prevalence of the Practice
Null if 1_mar_prac = 4.
When the text does not characterise prevalence, select the most plausible code and flag it.

| Code | Label | Apply when |
|------|-------|------------|
| 1 | widespread | General, common, normal, or universal in the population |
| 2 | rare_exception | Exceptional, rare, or limited to specific individuals |

When in doubt: if the practice is described as a custom or norm of a group → code 1.
If described as limited to a few individuals or an anomaly → code 2. Always flag.

---

## VARIABLE 3 — 3_pol_struc: Polygamy Structure
Null UNLESS 1_mar_prac = 2 or 3.
When 1_mar_prac = 3: code the POLYGAMOUS union only.
When the text confirms polygamy but gives no structural detail, pick best-fit and flag.

| Code | Label | Apply when |
|------|-------|------------|
| 1 | fixed_spouses | Normalised fixed number described (\"each man takes two wives\") |
| 2 | varied | Variable number with implied limits (\"richer man, more wives\") — always flag in 12_validation_flags |
| 3 | unlimited | No restrictions on number stated or implied |

When in doubt and text mentions harem or polygamy with no numeric detail → code 3 (unlimited) and flag.

---

## VARIABLE 4 — 4_ethnic_group: Ethnic Group(s) Named
STRING ARRAY — verbatim from text, no normalisation. Null if none named.
Preserve archaic transliterations exactly (e.g. \"Bechuanas\", \"Wanyamwesi\", \"Kafirs\").
Null if 1_mar_prac = 4.

---

## VARIABLE 5 — 5_soc_stat: Social Status of Practitioners
Null if 1_mar_prac = 4.
When the text does not specify social status, select best-fit and flag.

| Code | Label | Apply when |
|------|-------|------------|
| 1 | general_pop | Applies throughout the general population |
| 2 | elite_wealthy | Explicitly limited to elite, wealthy, noble, or chiefly individuals |
| 3 | eth_restric | Limited to a specific ethnic subgroup within a larger population |

When in doubt: if a group's custom is described without qualification → code 1 and flag.
If wealth or rank is mentioned as a condition → code 2 and flag.

---

## VARIABLE 6 — 6_obs_type: Type of Observation
Null if 1_mar_prac = 4.
When the source of the observation is ambiguous, select best-fit and flag.

| Code | Label | Apply when |
|------|-------|------------|
| 1 | auth_direct | Author explicitly states direct personal witnessing |
| 2 | auth_second_res | Named or described long-term resident of the area |
| 3 | auth_hearsay | Unnamed informant / \"it is said\" / \"they say\" / \"I was told\" |
| 4 | auth_gen | General statement with no attribution to a person or source |

When in doubt: travel-account narrative in first person without explicit witness claim → code 4 and flag.
\"It is said\" / \"I heard\" → code 3. Named resident informant → code 2.

---

## VARIABLE 7 — 7_obs_location: Geographic Location Present
Null if 1_mar_prac = 4.

| Code | Apply when |
|------|------------|
| 1 | A named geographic location or place identifier is present in the extract |
| 2 | No geographic location is present in the extract |

When a location is mentioned but its relation to the marriage practice is unclear → code 1
and populate 8_cited_loc with the name. Flag in 12_validation_flags if unsure of relevance.

---

## VARIABLE 8 — 8_cited_loc: Location Name(s)
STRING ARRAY — verbatim from text, no normalisation. Null if 7_obs_location != 1.
Preserve original spellings exactly (e.g. \"Timbuctoo\", \"the Zambeze\", \"Dahomey\").
List ALL named places, not just the primary one.

---

## VARIABLE 9 — 9_moral_op: Moral Opinion Present
Null if 1_mar_prac = 4.
When tone is ambiguous, select best-fit and flag.

| Code | Apply when |
|------|------------|
| 1 | moral_judge | Explicit moral, religious, or evaluative judgement about the practice |
| 2 | no_judge | No moral judgement present |

Indicators for code 1: (\"barbarous\"/\"degrading\"/\"sinful\"/\"uncivilised\"/\"revolting\"/\"commendable\") /
religious framing / comparative moral claim / desire for reform.
When in doubt about whether a remark is moral vs descriptive → code 2 and flag.

---

## VARIABLE 10 — 10_stance: Verbatim Moral Remark
STRING — verbatim from text. Null if 9_moral_op != 1.
Copy the exact phrase or sentence containing the judgement. Preserve original spelling and punctuation.

---

## VARIABLE 11 — 11_code_conf: Overall Coding Confidence
Always required. Null only if 1_mar_prac = 4.

| Code | Apply when |
|------|------------|
| 1 | high | >=95% confidence in accuracy of all coded variables |
| 2 | medium | 60-94% — one or more variables uncertain |
| 3 | low | <60% — substantial ambiguity throughout |

## 11_conf_note
String (<=30 words). Null only if 1_mar_prac = 4.
ALWAYS populate 11_conf_note (even when 11_code_conf = 1) when ANY of the following apply:
- Polyandry is present (one woman, multiple husbands)
- Behaviour appears exceptional, anomalous, or outside the norm for the described group
- The author explicitly marks the practice as unusual, rare, or surprising
- Mixed-race or cross-ethnic marriage is described
- Marriage involving religious figures, enslaved persons, or captives
- Any other case that seems like an edge case or exception to standard patterns
In these cases write a brief factual note describing what is unusual, e.g. \"polyandry: one wife shared by brothers\" or \"exceptional: European man marrying local woman against local custom\".

---

## VARIABLE 12 — 12_validation_flags: Per-Variable Human Validation Flags
Null if 1_mar_prac = 4. Otherwise always required — return [] if no flags.

Add a variable name to this array when ANY of the following apply:
- You are less than 70% confident in that specific variable's code
- You inferred the code from context rather than explicit textual evidence
- The row contains exceptional or anomalous behaviour (polyandry, cross-ethnic marriage, religious figures, enslaved persons, captives, author's own marriage, etc.) — in this case add \"EXCEPTIONAL\" as a string entry alongside any variable names

Do NOT flag variables that are null due to a conditional rule (those are correct nulls, not uncertain codes).

Examples:
- Uncertain about 0_subject and 5_soc_stat: \"12_validation_flags\": [\"0_subject\", \"5_soc_stat\"]
- Confident in everything: \"12_validation_flags\": []
- Row is not_applicable: \"12_validation_flags\": null

---

## LOGICAL CONSTRAINTS

| Condition | Required |
|-----------|----------|
| 1_mar_prac = 4 | ALL variables (incl. 0_subject) = null |
| 1_mar_prac = 1 | 3_pol_struc = null |
| 1_mar_prac = 2 or 3 | 3_pol_struc must have a value 1-3 |
| 1_mar_prac = 3 | 3_pol_struc refers to polygamous union ONLY |
| 7_obs_location != 1 | 8_cited_loc = null |
| 7_obs_location = 1 | 8_cited_loc must be a non-empty string array |
| 9_moral_op != 1 | 10_stance = null |
| 9_moral_op = 1 | 10_stance must be a non-empty string |

---

## EXAMPLES

Text: \"The Waswahili, like all Mahommedans, are allowed by their law to take four wives; but the rich alone can afford to maintain more than one or two. I found this custom degrading to the women.\"
Output: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 1, \"4_ethnic_group\": [\"Waswahili\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 2, \"8_cited_loc\": null, \"9_moral_op\": 1, \"10_stance\": \"I found this custom degrading to the women.\", \"11_code_conf\": 2, \"11_conf_note\": \"5_soc_stat: law universal but practice wealth-dependent\", \"12_validation_flags\": [\"5_soc_stat\"]}

Text: \"The Turks of this province maintain large harems, whilst the Christian Greeks of the same district take but a single wife.\"
Output: {\"0_subject\": 4, \"1_mar_prac\": 3, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Turks\", \"Greeks\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 2, \"8_cited_loc\": null, \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 2, \"11_conf_note\": \"0_subject ambiguous: Turks/Greeks may be local or foreign\", \"12_validation_flags\": [\"0_subject\"]}

Text: \"At Khartoum, my informant, a merchant long established in the city, assured me that among the Dongolawi only men of rank and fortune maintain more than one wife; the common people are practically monogamous from necessity.\"
Output: {\"0_subject\": 1, \"1_mar_prac\": 3, \"2_mar_diff\": 2, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Dongolawi\"], \"5_soc_stat\": 2, \"6_obs_type\": 2, \"7_obs_location\": 1, \"8_cited_loc\": [\"Khartoum\"], \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 1, \"11_conf_note\": null, \"12_validation_flags\": []}

Text: \"It is said that in the region between the Lualaba and the Lomami rivers, the Basonge practice a most revolting form of polygamy, in which a man may accumulate wives without limit.\"
Output: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Basonge\"], \"5_soc_stat\": 1, \"6_obs_type\": 3, \"7_obs_location\": 1, \"8_cited_loc\": [\"Lualaba\", \"Lomami\"], \"9_moral_op\": 1, \"10_stance\": \"a most revolting form of polygamy\", \"11_code_conf\": 1, \"11_conf_note\": null, \"12_validation_flags\": []}

Text: \"Among the Nairs of the Malabar coast, one woman is shared as wife by several brothers simultaneously.\"
Output: {\"0_subject\": 1, \"1_mar_prac\": 2, \"2_mar_diff\": 1, \"3_pol_struc\": 3, \"4_ethnic_group\": [\"Nairs\"], \"5_soc_stat\": 1, \"6_obs_type\": 4, \"7_obs_location\": 1, \"8_cited_loc\": [\"Malabar\"], \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 1, \"11_conf_note\": \"polyandry: one woman shared as wife by several brothers simultaneously\", \"12_validation_flags\": [\"EXCEPTIONAL\"]}

Text: \"CHAPTER IV. Of the customs of the Kafirs — p. 47. Of the trade in ivory — p. 63.\"
Output: {\"0_subject\": null, \"1_mar_prac\": 4, \"2_mar_diff\": null, \"3_pol_struc\": null, \"4_ethnic_group\": null, \"5_soc_stat\": null, \"6_obs_type\": null, \"7_obs_location\": null, \"8_cited_loc\": null, \"9_moral_op\": null, \"10_stance\": null, \"11_code_conf\": null, \"11_conf_note\": null, \"12_validation_flags\": null}

Text: \"He returned home to his wife after many months at sea.\"
Note: incidental mention only — should have been filtered at Stage 1, but if it passes:
Output: {\"0_subject\": 4, \"1_mar_prac\": 1, \"2_mar_diff\": 1, \"3_pol_struc\": null, \"4_ethnic_group\": null, \"5_soc_stat\": 1, \"6_obs_type\": 1, \"7_obs_location\": 2, \"8_cited_loc\": null, \"9_moral_op\": 2, \"10_stance\": null, \"11_code_conf\": 3, \"11_conf_note\": \"incidental mention only; all codes inferred with very low confidence\", \"12_validation_flags\": [\"0_subject\", \"1_mar_prac\", \"2_mar_diff\", \"5_soc_stat\", \"6_obs_type\"]}

---

## OUTPUT FORMAT
Return ONLY a single-line valid JSON object. No markdown, no explanation, no text before or after.
{\"0_subject\": X, \"1_mar_prac\": X, \"2_mar_diff\": X, \"3_pol_struc\": X, \"4_ethnic_group\": X, \"5_soc_stat\": X, \"6_obs_type\": X, \"7_obs_location\": X, \"8_cited_loc\": X, \"9_moral_op\": X, \"10_stance\": X, \"11_code_conf\": X, \"11_conf_note\": X, \"12_validation_flags\": X}"""


print(f'✓ Stage 1 prompt : {len(STAGE_1_PROMPT):,} characters')
print(f'✓ Stage 2 prompt : {len(STAGE_2_PROMPT):,} characters')
print()
print('v4 key rules:')
print('  No unclear/unspecified codes — LLM always commits to best-fit + flags if <70% confident')
print('  0_subject: 4 codes (no code 5 unclear)')
print('  2_mar_diff: 2 codes (no code 3 unclear)')
print('  3_pol_struc: 4 codes (no code 5 unspecified)')
print('  5_soc_stat: 3 codes (no code 4 unspecified)')
print('  6_obs_type: 4 codes (no code 5 unclear)')
print('  7_obs_location: 2 codes (no code 3 unclear)')
print('  9_moral_op: 2 codes (no code 3 unclear)')

✓ Stage 1 prompt : 1,573 characters
✓ Stage 2 prompt : 14,094 characters

v4 key rules:
  No unclear/unspecified codes — LLM always commits to best-fit + flags if <70% confident
  0_subject: 4 codes (no code 5 unclear)
  2_mar_diff: 2 codes (no code 3 unclear)
  3_pol_struc: 4 codes (no code 5 unspecified)
  5_soc_stat: 3 codes (no code 4 unspecified)
  6_obs_type: 4 codes (no code 5 unclear)
  7_obs_location: 2 codes (no code 3 unclear)
  9_moral_op: 2 codes (no code 3 unclear)


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — IMPORTS, CLASSIFIER & PROGRESS TRACKER
# ════════════════════════════════════════════════════════

import json
import re
import time
import pandas as pd
import anthropic
from pathlib import Path
from datetime import datetime


class ProgressTracker:
    """Saves each row to Drive immediately. Resume-safe on interruption."""

    def __init__(self, output_dir: Path, document_name: str):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.document_name = document_name
        self.temp_dir = self.output_dir / f'temp_{document_name}'
        self.temp_dir.mkdir(exist_ok=True)
        self.checkpoint_file = self.temp_dir / 'checkpoint.json'
        self.progress = self._load_checkpoint()

    def _load_checkpoint(self) -> dict:
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file) as f:
                return json.load(f)
        return {'completed_rows': [], 'start_time': datetime.now().isoformat()}

    def _save_checkpoint(self):
        self.progress['last_update'] = datetime.now().isoformat()
        with open(self.checkpoint_file, 'w') as f:
            json.dump(self.progress, f, indent=2)

    def should_process(self, row_number: int) -> bool:
        return row_number not in self.progress['completed_rows']

    def save_row(self, row_number: int, result: dict):
        path = self.temp_dir / f'row_{row_number:06d}.json'
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(
                {'row_number': row_number, 'result': result,
                 'processed_at': datetime.now().isoformat()},
                f, indent=2, ensure_ascii=False
            )
        if row_number not in self.progress['completed_rows']:
            self.progress['completed_rows'].append(row_number)
        self._save_checkpoint()

    def load_all_results(self) -> list:
        results = []
        for path in sorted(self.temp_dir.glob('row_*.json')):
            with open(path, encoding='utf-8') as f:
                results.append(json.load(f)['result'])
        return results

    def finalize(self, all_results: list) -> Path:
        output_path = self.output_dir / f'{self.document_name}.json'
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(
                {'model': MODEL, 'codebook_version': 'v5.0',
                 'document_name': self.document_name,
                 'total_rows': len(all_results),
                 'processed_at': datetime.now().isoformat(),
                 'results': all_results},
                f, indent=2, ensure_ascii=False
            )
        print(f'\n✅ Results saved to Drive: {output_path}')
        return output_path


class TwoStageClassifier:
    """
    Two-stage pipeline — Codebook v4.

    Key v4 changes:
      - No unclear/unspecified codes. LLM always picks best-fit + flags.
      - 0_subject: codes 1-4 only (no code 5)
      - 2_mar_diff: codes 1-2 only (no code 3)
      - 3_pol_struc: codes 1-4 only (no code 5)
      - 5_soc_stat: codes 1-3 only (no code 4)
      - 6_obs_type: codes 1-4 only (no code 5)
      - 7_obs_location: codes 1-2 only (no code 3)
      - 9_moral_op: codes 1-2 only (no code 3)
      - max_tokens increased to 650 to handle fuller 12_validation_flags arrays
    """

    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.stats = {
            'total_rows': 0, 'filtered': 0, 'classified': 0,
            'not_applicable': 0, 'flagged': 0, 'errors': 0,
            'stage1_in': 0, 'stage1_out': 0, 'stage1_cache': 0,
            'stage2_in': 0, 'stage2_out': 0, 'stage2_cache': 0,
        }

    def _call(self, system_prompt: str, text: str, max_tokens: int,
              stat_prefix: str, max_retries: int = 3) -> dict:
        for attempt in range(max_retries):
            try:
                response = self.client.messages.create(
                    model=MODEL,
                    max_tokens=max_tokens,
                    temperature=TEMPERATURE,
                    system=[{'type': 'text', 'text': system_prompt,
                             'cache_control': {'type': 'ephemeral'}}],
                    messages=[{'role': 'user', 'content': text}]
                )
                u = response.usage
                self.stats[f'{stat_prefix}_in']    += getattr(u, 'input_tokens', 0)
                self.stats[f'{stat_prefix}_out']   += getattr(u, 'output_tokens', 0)
                self.stats[f'{stat_prefix}_cache'] += getattr(u, 'cache_read_input_tokens', 0)
                return {'text': response.content[0].text.strip()}
            except anthropic.RateLimitError:
                if attempt < max_retries - 1:
                    wait = 60 * (2 ** attempt)
                    print(f'   ⚠️  Rate limit — waiting {wait}s…')
                    time.sleep(wait)
                else:
                    return {'error': 'Rate limit exceeded'}
            except Exception as e:
                if attempt < max_retries - 1:
                    wait = 10 * (2 ** attempt)
                    print(f'   ⚠️  Retry {attempt+1} in {wait}s: {e}')
                    time.sleep(wait)
                else:
                    return {'error': str(e)}

    def _parse_json(self, raw: str) -> dict:
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
        json_str = m.group(1) if m else re.search(r'\{.*\}', raw, re.DOTALL).group(0)
        return json.loads(json_str)

    def process_row(self, text: str, row_number: int) -> dict:
        self.stats['total_rows'] += 1

        r1 = self._call(STAGE_1_PROMPT, text, 60, 'stage1')
        if 'error' in r1:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error', 'error': r1['error']}

        try:
            f = json.loads(r1['text'])
        except Exception:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error',
                    'error': f'Stage 1 JSON parse failed: {r1["text"][:80]}'}

        if not f.get('applicable', False):
            self.stats['filtered'] += 1
            return {'row_number': row_number, 'stage': 'filtered',
                    'reason': f.get('reason', 'not applicable')}

        r2 = self._call(STAGE_2_PROMPT, text, 650, 'stage2')
        if 'error' in r2:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error', 'error': r2['error']}

        try:
            c = self._parse_json(r2['text'])
        except Exception:
            self.stats['errors'] += 1
            return {'row_number': row_number, 'stage': 'error',
                    'error': f'Stage 2 JSON parse failed: {r2["text"][:80]}'}

        mp    = c.get('1_mar_prac')
        flags = c.get('12_validation_flags') or []

        if mp == 4:
            self.stats['not_applicable'] += 1
            stage = 'not_applicable'
        else:
            self.stats['classified'] += 1
            stage = 'classified'

        if isinstance(flags, list) and len(flags) > 0:
            self.stats['flagged'] += 1

        return {'row_number': row_number, 'stage': stage, 'classification': c}

    def print_stats(self):
        s = self.stats
        n = s['total_rows'] or 1
        for stage, label in [('stage1', 'Stage 1'), ('stage2', 'Stage 2')]:
            total = s[f'{stage}_in'] + s[f'{stage}_cache']
            hit   = (s[f'{stage}_cache'] / total * 100) if total else 0
            s[f'{stage}_hit'] = hit
        print(f'\n{"="*58}')
        print(f'PIPELINE STATS  ({MODEL}  |  Codebook v4)')
        print(f'{"="*58}')
        print(f'Total rows processed    : {s["total_rows"]:,}')
        print(f'Stage 1 filtered        : {s["filtered"]:,}  ({s["filtered"]/n*100:.1f}%)')
        print(f'Stage 2 classified      : {s["classified"]:,}')
        print(f'Stage 2 not_applicable  : {s["not_applicable"]:,}  (1_mar_prac=4)')
        print(f'Rows flagged for review : {s["flagged"]:,}  (12_validation_flags non-empty)')
        print(f'Errors                  : {s["errors"]:,}')
        print(f'---')
        print(f'Stage 1 tokens in/out/cached : {s["stage1_in"]:,} / {s["stage1_out"]:,} / {s["stage1_cache"]:,}  (hit: {s["stage1_hit"]:.1f}%)')
        print(f'Stage 2 tokens in/out/cached : {s["stage2_in"]:,} / {s["stage2_out"]:,} / {s["stage2_cache"]:,}  (hit: {s["stage2_hit"]:.1f}%)')
        print(f'{"="*58}')


print('✓ ProgressTracker and TwoStageClassifier (v4) defined')

✓ ProgressTracker and TwoStageClassifier (v4) defined


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — LOAD DATA
# ════════════════════════════════════════════════════════

df = pd.read_excel(INPUT_PATH)
if MAX_ROWS:
    df = df.head(MAX_ROWS)

assert TEXT_COLUMN in df.columns, (
    f"Column '{TEXT_COLUMN}' not found. Available: {list(df.columns)}"
)

print(f'✓ Data loaded: {len(df)} rows')
print(f'  Columns: {list(df.columns)}')
print(f'\nFirst text preview ({len(str(df.iloc[0][TEXT_COLUMN]))} chars):')
print(str(df.iloc[0][TEXT_COLUMN])[:400], '...')

✓ Data loaded: 50 rows
  Columns: ['item_ID', 'guid_hash', 'page_number', 'page_type', 'full_text', 'translate', 'tokens', 'english_words', 'search_query', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']

First text preview (1481 chars):
2.00 NEW VOYAGE IN EGYPT, This is the famous river of Adonis, so well known for the vines that line its shores, and for the charming description that Milton has left us of it. Antoura, built amidst mulberry trees, presents a cheerful aspect but offers no remarkable features. Near this village, one can see a convent of nuns where the wife of Mr. Wortley Montague is staying. The Christians of this ( ...


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — RUN TWO-STAGE PIPELINE
# ════════════════════════════════════════════════════════

print('='*60)
print('MARRIAGE PRACTICE CLASSIFIER — TWO-STAGE PIPELINE  v5')
print('='*60)
print(f'  Model: {MODEL}  |  Codebook: v5.0  |  Rows: {len(df)}')

classifier = TwoStageClassifier(ANTHROPIC_API_KEY)
tracker    = ProgressTracker(Path(OUTPUT_DIR), 'haiku_classifications_v4')

completed = len(tracker.progress['completed_rows'])
if completed > 0:
    print(f'\n   ℹ️  Resuming: {completed}/{len(df)} rows already processed')

start = time.time()

for idx, row in df.iterrows():
    row_number = idx + 1
    if not tracker.should_process(row_number):
        continue

    text = str(row[TEXT_COLUMN])
    print(f'\n🔍 Row {row_number}/{len(df)}  ({len(text)} chars)')

    result = classifier.process_row(text, row_number)

    if result['stage'] == 'filtered':
        print(f'   ⏩ FILTERED — {result["reason"]}')
    elif result['stage'] == 'not_applicable':
        print(f'   🔇 NOT APPLICABLE (1_mar_prac=4)')
    elif result['stage'] == 'classified':
        c     = result['classification']
        flags = c.get('12_validation_flags') or []
        flag_str = f'  ⚑ REVIEW: {flags}' if flags else ''
        print(
            f'   ✓  subj={c.get("0_subject")}  '
            f'mar={c.get("1_mar_prac")}  '
            f'pol={c.get("3_pol_struc")}  '
            f'moral={c.get("9_moral_op")}  '
            f'conf={c.get("11_code_conf")}'
            f'{flag_str}'
        )
        if c.get('11_conf_note'):
            print(f'   📝 {c["11_conf_note"]}')
    else:
        print(f'   ❌ ERROR — {result.get("error")}')

    tracker.save_row(row_number, result)

    if row_number < len(df):
        time.sleep(REQUEST_DELAY)

    if row_number % BATCH_SIZE == 0:
        elapsed   = time.time() - start
        rate      = row_number / elapsed
        remaining = (len(df) - row_number) / rate if rate > 0 else 0
        print(f'\n   📊 {row_number}/{len(df)} ({row_number/len(df)*100:.1f}%) '
              f'| {rate:.2f} rows/sec | ~{remaining/60:.1f} min remaining')

classifier.print_stats()

print('\n📥 Consolidating results...')
all_results = tracker.load_all_results()
output_path = tracker.finalize(all_results)

MARRIAGE PRACTICE CLASSIFIER — TWO-STAGE PIPELINE  v5
  Model: claude-haiku-4-5-20251001  |  Codebook: v5.0  |  Rows: 50

🔍 Row 1/50  (1481 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": true}
```

🔍 Row 2/50  (2946 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": false, "reason": "No marriage-related content; text is a 

🔍 Row 3/50  (2220 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": true}
```

🔍 Row 4/50  (1414 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": true}
```

🔍 Row 5/50  (2065 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": false, "reason": "Text discusses ecclesiastical practices

🔍 Row 6/50  (433 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": false, "reason": "incidental mention of 'wife' with no st

🔍 Row 7/50  (1894 chars)
   ❌ ERROR — Stage 1 JSON parse failed: ```json
{"applicable": false, "reason": "No substantive marriage content; in

KeyboardInterrupt: 

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — FLATTEN, SUMMARISE & EXPORT CSVs
# ════════════════════════════════════════════════════════

# Valid codes per variable (v4 — no unclear/unspecified)
VALID_CODES = {
    '0_subject':      {1: 'african_african', 2: 'african_african_diff',
                       3: 'african_foreign', 4: 'foreign_foreign'},
    '1_mar_prac':     {1: 'monogamy_only', 2: 'polygamy_only',
                       3: 'both', 4: 'not_applicable'},
    '2_mar_diff':     {1: 'widespread', 2: 'rare_exception'},
    '3_pol_struc':    {1: 'fixed_spouses', 2: 'varied', 3: 'unlimited'},
    '5_soc_stat':     {1: 'general_pop', 2: 'elite_wealthy', 3: 'eth_restric'},
    '6_obs_type':     {1: 'auth_direct', 2: 'auth_second_res',
                       3: 'auth_hearsay', 4: 'auth_gen'},
    '7_obs_location': {1: 'yes', 2: 'no'},
    '9_moral_op':     {1: 'moral_judge', 2: 'no_judge'},
    '11_code_conf':   {1: 'high', 2: 'medium', 3: 'low'},
}

# Flatten all coded rows
rows = []
for r in all_results:
    if r.get('stage') in ('classified', 'not_applicable') and r.get('classification'):
        entry = {'row_number': r['row_number'], 'stage': r['stage']}
        entry.update(r['classification'])
        flags = entry.get('12_validation_flags')
        entry['12_validation_flags'] = '|'.join(flags) if isinstance(flags, list) else ''
        rows.append(entry)

df_coded    = pd.DataFrame(rows)
df_filtered = pd.DataFrame([
    {'row_number': r['row_number'], 'reason': r.get('reason', '')}
    for r in all_results if r.get('stage') == 'filtered'
])
df_flagged  = df_coded[
    df_coded['12_validation_flags'].str.len() > 0
].copy() if '12_validation_flags' in df_coded.columns else pd.DataFrame()

# Check for out-of-codebook codes
print('\n── Out-of-codebook code check ──────────────────────────')
for var, valid in VALID_CODES.items():
    if var not in df_coded.columns:
        continue
    col = pd.to_numeric(df_coded[var], errors='coerce')
    bad = col.dropna()[~col.dropna().isin(valid.keys())]
    if len(bad) > 0:
        print(f'  ⚠️  {var}: unexpected codes {bad.unique().tolist()} ({len(bad)} rows)')
    else:
        print(f'  ✓  {var}')

# Frequency tables
print(f'\nCoded rows: {len(df_coded)}  |  Filtered: {len(df_filtered)}  |  Flagged: {len(df_flagged)}')
for var, codes in VALID_CODES.items():
    if var not in df_coded.columns:
        continue
    print(f'\n── {var} ' + '─' * max(1, 42 - len(var)))
    vc = df_coded[var].value_counts(dropna=False).sort_index()
    for code, count in vc.items():
        pct   = count / len(df_coded) * 100
        label = codes.get(code, 'null' if pd.isna(code) else f'UNEXPECTED:{code}')
        marker = ' ⚠️' if isinstance(label, str) and 'UNEXPECTED' in label else ''
        print(f'  {str(code):<4} {label:<22} {count:>5}  ({pct:.1f}%){marker}')

# Most flagged variables
if len(df_flagged) > 0:
    all_flags = []
    for s in df_flagged['12_validation_flags']:
        if s: all_flags.extend(s.split('|'))
    print(f'\n── Most flagged variables ──────────────────────────')
    print(pd.Series(all_flags).value_counts().to_string())

# Stage 1 filter reasons
if len(df_filtered) > 0:
    print(f'\n── Stage 1 filter reasons ──────────────────────────')
    print(df_filtered['reason'].value_counts().to_string())

# Export
out = Path(OUTPUT_DIR)
classified_csv = out / 'haiku_v4_coded.csv'
filtered_csv   = out / 'haiku_v4_filtered.csv'
flagged_csv    = out / 'haiku_v4_flagged_for_review.csv'

df_coded.to_csv(classified_csv, index=False)
df_filtered.to_csv(filtered_csv, index=False)
if len(df_flagged) > 0:
    df_flagged.to_csv(flagged_csv, index=False)

print(f'\n✅ Coded CSV             : {classified_csv}')
print(f'✅ Filtered CSV          : {filtered_csv}')
print(f'✅ Flagged-for-review CSV: {flagged_csv}')
print(f'\n→ Run Cell 8 to review flagged rows interactively.')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — HUMAN-IN-THE-LOOP REVIEW
#
# Steps through every flagged row one at a time.
# For each flagged variable the reviewer can:
#   • Accept the LLM code as-is
#   • Override with the correct code
# Saves a reconciled CSV: haiku_v4_reconciled.csv
#   with columns <var>_llm  and  <var>_final  for every flagged variable.
# Skips rows where 12_validation_flags is empty (nothing to review).
# ════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Codebook labels (v4) ─────────────────────────────────────────────────
VALID_CODES = {
    '0_subject':      {1: 'african_african', 2: 'african_african_diff',
                       3: 'african_foreign', 4: 'foreign_foreign'},
    '1_mar_prac':     {1: 'monogamy_only', 2: 'polygamy_only',
                       3: 'both', 4: 'not_applicable'},
    '2_mar_diff':     {1: 'widespread', 2: 'rare_exception'},
    '3_pol_struc':    {1: 'fixed_spouses', 2: 'varied', 3: 'unlimited'},
    '5_soc_stat':     {1: 'general_pop', 2: 'elite_wealthy', 3: 'eth_restric'},
    '6_obs_type':     {1: 'auth_direct', 2: 'auth_second_res',
                       3: 'auth_hearsay', 4: 'auth_gen'},
    '7_obs_location': {1: 'yes', 2: 'no'},
    '9_moral_op':     {1: 'moral_judge', 2: 'no_judge'},
}

ALL_CODED_VARS = list(VALID_CODES.keys())  # only numeric coded vars are reviewable

# ── Load data ─────────────────────────────────────────────────────────────
flagged_csv    = Path(OUTPUT_DIR) / 'haiku_v4_flagged_for_review.csv'
reconciled_csv = Path(OUTPUT_DIR) / 'haiku_v4_reconciled.csv'

if not flagged_csv.exists():
    print('No flagged rows file found. Run Cells 6–7 first.')
else:
    df_review = pd.read_csv(flagged_csv)

    # Load original texts to display alongside codes
    df_texts = pd.read_excel(INPUT_PATH)[[TEXT_COLUMN]].reset_index()
    df_texts.columns = ['row_index', TEXT_COLUMN]
    df_texts['row_number'] = df_texts['row_index'] + 1

    # Merge texts into review frame
    df_review = df_review.merge(df_texts[['row_number', TEXT_COLUMN]],
                                on='row_number', how='left')

    # Initialise _final columns from LLM codes
    for var in ALL_CODED_VARS:
        if var in df_review.columns:
            df_review[f'{var}_llm']   = df_review[var]
            df_review[f'{var}_final'] = df_review[var]  # will be overwritten by reviewer

    df_review['review_status'] = 'pending'
    df_review['reviewer_note'] = ''

    # ── Widget state ─────────────────────────────────────────────────────
    state = {
        'idx': 0,
        'total': len(df_review),
        'decisions': {},   # row_number → {var: final_code}
    }

    out_display = widgets.Output()

    def render_row(idx):
        """Render the review widget for one flagged row."""
        out_display.clear_output(wait=True)

        if idx >= state['total']:
            with out_display:
                print('\n🎉 All flagged rows reviewed!')
                print(f'   Saving reconciled output to {reconciled_csv}...')
                save_reconciled()
            return

        row = df_review.iloc[idx]
        flags_str = str(row.get('12_validation_flags', ''))
        flagged_vars = [v.strip() for v in flags_str.split('|') if v.strip()]
        flagged_vars = [v for v in flagged_vars if v in VALID_CODES]

        row_num   = int(row['row_number'])
        text_full = str(row.get(TEXT_COLUMN, ''))
        text_show = text_full[:600] + ('…' if len(text_full) > 600 else '')

        with out_display:
            print(f'\n{"─"*65}')
            print(f'ROW {row_num}  [{idx+1} of {state["total"]}]  '
                  f'  1_mar_prac={row.get("1_mar_prac")}  '
                  f'conf={row.get("11_code_conf")}')
            if row.get('11_conf_note'):
                print(f'LLM note: {row["11_conf_note"]}')
            print(f'{"─"*65}')
            print(f'TEXT:\n{text_show}')
            print(f'{"─"*65}')
            print(f'FLAGGED VARIABLES: {flagged_vars}')
            print()

            # Show all LLM codes for context
            for var in ALL_CODED_VARS:
                if var in df_review.columns:
                    code = row.get(var)
                    label = VALID_CODES[var].get(code, str(code)) if pd.notna(code) else 'null'
                    flag_marker = ' ⚑' if var in flagged_vars else ''
                    print(f'  {var:<20} {str(code) if pd.notna(code) else "null":<4} {label}{flag_marker}')

        # Build one dropdown per flagged variable
        dropdowns = {}
        for var in flagged_vars:
            codes = VALID_CODES[var]
            current_code = row.get(var)
            options = [(f'{k} — {v}', k) for k, v in codes.items()]
            current_val = int(current_code) if pd.notna(current_code) else list(codes.keys())[0]
            dd = widgets.Dropdown(
                options=options,
                value=current_val,
                description=f'{var}:',
                style={'description_width': '160px'},
                layout=widgets.Layout(width='480px')
            )
            dropdowns[var] = dd

        note_box = widgets.Text(
            value='',
            placeholder='Optional reviewer note (press Enter or click Save to submit)',
            description='Note:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='560px')
        )

        btn_save = widgets.Button(
            description='Save & Next ▶',
            button_style='success',
            layout=widgets.Layout(width='160px')
        )
        btn_skip = widgets.Button(
            description='Skip (keep LLM codes)',
            button_style='warning',
            layout=widgets.Layout(width='200px')
        )

        progress_bar = widgets.IntProgress(
            value=idx, min=0, max=state['total'],
            description=f'{idx}/{state["total"]}',
            bar_style='info',
            layout=widgets.Layout(width='400px')
        )

        def on_save(_):
            decisions = {var: dd.value for var, dd in dropdowns.items()}
            state['decisions'][row_num] = decisions

            # Write decisions into df_review
            for var, final_code in decisions.items():
                df_review.at[idx, f'{var}_final'] = final_code
            df_review.at[idx, 'review_status'] = 'reviewed'
            df_review.at[idx, 'reviewer_note'] = note_box.value

            state['idx'] += 1
            render_row(state['idx'])

        def on_skip(_):
            # Keep LLM codes as final
            df_review.at[idx, 'review_status'] = 'skipped'
            state['idx'] += 1
            render_row(state['idx'])

        btn_save.on_click(on_save)
        btn_skip.on_click(on_skip)

        widget_rows = list(dropdowns.values()) + [note_box,
                           widgets.HBox([btn_save, btn_skip]),
                           progress_bar]
        display(widgets.VBox(widget_rows))

    def save_reconciled():
        """Write the reconciled CSV — one row per flagged row,
           with both LLM and final codes for reviewable variables."""
        # Build output: keep all original columns + add _llm and _final for coded vars
        out_cols = ['row_number', 'stage', 'review_status', 'reviewer_note',
                    '1_mar_prac', '11_code_conf', '11_conf_note',
                    '12_validation_flags', TEXT_COLUMN]
        for var in ALL_CODED_VARS:
            if f'{var}_llm' in df_review.columns:
                out_cols += [f'{var}_llm', f'{var}_final']
        # add string vars
        for sv in ['4_ethnic_group', '8_cited_loc', '10_stance']:
            if sv in df_review.columns:
                out_cols.append(sv)

        out_cols = [c for c in out_cols if c in df_review.columns]
        df_review[out_cols].to_csv(reconciled_csv, index=False)

        with out_display:
            reviewed = (df_review['review_status'] == 'reviewed').sum()
            skipped  = (df_review['review_status'] == 'skipped').sum()
            pending  = (df_review['review_status'] == 'pending').sum()
            print(f'\n✅ Reconciled CSV saved: {reconciled_csv}')
            print(f'   Reviewed  : {reviewed}')
            print(f'   Skipped   : {skipped}')
            print(f'   Pending   : {pending}')
            print('\n   Open haiku_v4_reconciled.csv in Google Sheets.')
            print('   The <var>_final columns are your gold-standard codes.')

    # ── Save button (mid-session emergency save) ──────────────────────────
    btn_save_now = widgets.Button(
        description='💾 Save progress now',
        button_style='info',
        layout=widgets.Layout(width='200px')
    )
    btn_save_now.on_click(lambda _: save_reconciled())

    if state['total'] == 0:
        print('✓ No flagged rows to review — all LLM codes are high-confidence.')
    else:
        print(f'Starting review of {state["total"]} flagged rows.')
        print('For each row: review flagged variables, adjust if needed, then Save & Next.')
        print('Click "💾 Save progress now" at any time to write a partial reconciled CSV.\n')
        display(btn_save_now)
        display(out_display)
        render_row(0)